# Benchmark 1: 2 Class vs 4 Class: Cross-Session

In [1]:
import numpy as np
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery, LeftRightImagery
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from mne.decoding import CSP
from moabb.evaluations import CrossSubjectEvaluation
from sklearn.pipeline import make_pipeline
from scipy import signal
from scipy.io import loadmat
import os
import mne
from scipy.linalg import logm, expm
from sklearn.svm import SVC
from sklearn.metrics import balanced_accuracy_score
from pyriemann.estimation import Covariances
import scipy

In [2]:
from pyriemann.utils import mean_riemann
from scipy.optimize import minimize
from pymanopt import Problem
from pymanopt.manifolds import SpecialOrthogonalGroup
from pymanopt.optimizers import SteepestDescent
from pymanopt import Problem
from functools import partial
from pymanopt.function import numpy as pymanopt_numpy
import autograd.numpy as anp  # Autograd's NumPy replacement
from autograd import grad
import pymanopt
import autograd.scipy.linalg as linalg
from pyriemann.classification import MDM

In [3]:
def logm_approx(A):
    I = anp.eye(A.shape[0])  # Identity matrix
    return A - I - 0.5 * (A - I) @ (A - I) 

def frobenius_norm(X):
    return anp.sqrt(anp.sum(X**2))


In [4]:
active_all_event_ids = {'769': 769, '770': 770, '771': 771, '772': 772}
active_lr_event_ids = {'769': 769, '770': 770}
unknown_event_id = {'783': 783} 

In [5]:
data_dir = '/home/vishwa/eeg_tl/Recreating papers/BCICIV_2a'

In [6]:
def encode_labels(labels_list):
    encoded_list = []
    for labels in labels_list:
        # Create mapping from original labels to 0,1
        unique_labels = np.unique(labels)
        label_map = {unique_labels[i]: i for i in range(len(unique_labels))}
        
        # Apply mapping
        encoded = np.array([label_map[label] for label in labels])
        encoded_list.append(encoded)
    return encoded_list

In [7]:
# Define a causal bandpass filter function using a Butterworth design.
def causal_bandpass_filter(data, lowcut=8, highcut=30, fs=250, order=5):
    nyq = 0.5 * fs
    # Normalize the cutoff frequencies (Matlab's fir1 expects normalized cutoff frequencies
    low = lowcut / nyq
    high = highcut / nyq
    # Design the FIR filter. Note: order+1 coefficients are returned to match Matlab's fir1 which returns n+1 taps.
    b = signal.firwin(order + 1, [low, high], window='hamming', pass_zero=False)
    # Apply the filter causally using lfilter (this introduces a constant delay).
    filtered_data = signal.lfilter(b, [1.0], data)
    return filtered_data

In [8]:
# With a sampling frequency of 250 Hz, 1001 samples equate to 1001/250 seconds.
sfreq = 250
# tmin = 0.5       # Epoch start at cue onset.
# # Set tmax so that n_samples = (tmax-tmin)*sfreq + 1 = 1001, i.e. 4 seconds long.
# tmax = 3.5  # This gives 4.0 seconds.
filter_order = 50

In [9]:
pairs = [
    (0.5, 3.5),
    (0.5, 2.5),
    (0.5, 1.5),
    (1.5, 3.5),
    (1.5, 2.5),
    (2.5, 3.5)
]

In [10]:
def twofour_crosssession(n_classes):
    for tmin, tmax in pairs:
        if(n_classes==2):
            event_ids = active_lr_event_ids
        else:
            event_ids = active_all_event_ids
        

        train_active_X = []         # List to hold numpy arrays with shape (n_trials, 22, 1001) per subject.
        train_active_y = []         # List to hold event labels per subject.
        train_active_metadata = []  # List to hold event metadata per subject.

        # Loop over subjects. Assume files are named "A01T.gdf", "A02T.gdf", ..., "A09T.gdf".
        for subj in range(1, 10):
            filename = os.path.join(data_dir, f'A{subj:02d}T.gdf')
            
            # Read the GDF file (using preload=True to load data into memory).
            train_raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False, eog=['EOG-left', 'EOG-central', 'EOG-right'])
            # Retain only EEG channels (22 channels) and exclude EOG channels.
            train_raw.pick_types(eeg=True, eog=False)
            
            # Extract events corresponding only to the four desired types.
            train_active_events, _ = mne.events_from_annotations(train_raw, event_id=event_ids)
            
            # Create epochs from tmin to tmax.
            train_active_epochs = mne.Epochs(train_raw, train_active_events, event_id=event_ids, tmin=tmin, tmax=tmax,
                                baseline=None, preload=True, verbose=False)
            
            # Get the epoch data (num_epochs x 22 channels x 1001 samples).
            train_active_data = train_active_epochs.get_data()
            
            # Print the number of extracted epochs to verify
            print(f"Subject {subj}: Epoch data shape {train_active_data.shape}")
            
            # Sampling frequency from raw.info (should be 250).
            fs = int(train_raw.info['sfreq'])
            # print(fs)
            n_trials, n_channels, n_times = train_active_data.shape
            train_active_filtered_data = np.empty_like(train_active_data)
            
            # Apply the causal bandpass filter channel‐wise for each trial.
            for trial in range(n_trials):
                for ch in range(n_channels):
                    train_active_filtered_data[trial, ch, :] = causal_bandpass_filter(
                        train_active_data[trial, ch, :],
                        lowcut=8,    # Lower bound of sensorimotor rhythm.
                        highcut=30,  # Upper bound of sensorimotor rhythm.
                        fs=fs,
                        order=filter_order     # Lower order for a smoother causal filter.
                    )
            # Apply the causal bandpass filter channel‐wise for each trial.
            # for trial in range(n_trials):
            #     for ch in range(n_channels):
            #         train_active_filtered_data[trial, ch, :] = train_active_data[trial, ch, :]
            
            # Append the processed data, labels, and event metadata.
            train_active_X.append(train_active_filtered_data)
            train_active_y.append(train_active_epochs.events[:, 2])  # The third column holds the event code.
            train_active_metadata.append(train_active_epochs.events)

        print("Loaded data for", len(train_active_X), "subjects.")

        eval_active_X = []         
        eval_active_y = []        
        eval_active_metadata = [] 

        # Loop over subjects. Assume files are named "A01T.gdf", "A02T.gdf", ..., "A09T.gdf".
        for subj in range(1, 10):
            filename = os.path.join(data_dir, f'A{subj:02d}E.gdf')
            mat_data = loadmat(f'/home/vishwa/eeg_tl/Recreating papers/BCICIV_2A true labels/A{subj:02d}E.mat')
            true_y =  np.array(mat_data['classlabel'], dtype=np.int64).reshape(288,) + 768
            # Read the GDF file (using preload=True to load data into memory).
            eval_raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False, eog=['EOG-left', 'EOG-central', 'EOG-right'])
            # Retain only EEG channels (22 channels) and exclude EOG channels.
            eval_raw.pick_types(eeg=True, eog=False)
            
            # Extract events corresponding only to the four desired types.
            eval_active_events, _ = mne.events_from_annotations(eval_raw, event_id=unknown_event_id)
            
            # Create epochs from tmin to tmax.
            eval_active_epochs = mne.Epochs(eval_raw, eval_active_events, event_id=unknown_event_id, tmin=tmin, tmax=tmax,
                                baseline=None, preload=True, verbose=False)
            
            # Get the epoch data (num_epochs x 22 channels x 1001 samples).
            eval_active_data = eval_active_epochs.get_data()
            
            # Print the number of extracted epochs to verify
            print(f"Subject {subj}: Epoch data shape {eval_active_data.shape}")
            
            # Sampling frequency from raw.info (should be 250).
            fs = int(eval_raw.info['sfreq'])
            # print(fs)
            n_trials, n_channels, n_times = eval_active_data.shape
            eval_active_filtered_data = np.empty_like(eval_active_data)
            
            # Apply the causal bandpass filter channel‐wise for each trial.
            for trial in range(n_trials):
                for ch in range(n_channels):
                    eval_active_filtered_data[trial, ch, :] = causal_bandpass_filter(
                        eval_active_data[trial, ch, :],
                        lowcut=8,    # Lower bound of sensorimotor rhythm.
                        highcut=30,  # Upper bound of sensorimotor rhythm.
                        fs=fs,
                        order=filter_order     # Lower order for a smoother causal filter.
                    )
            
            # for trial in range(n_trials):
            #     for ch in range(n_channels):
            #         eval_active_filtered_data[trial, ch, :] = eval_active_data[trial, ch, :]
                        
            
            # Append the processed data, labels, and event metadata.
            eval_active_X.append(eval_active_filtered_data)
            eval_active_y.append(true_y)  # The third column holds the event code.
            eval_active_metadata.append(eval_active_epochs.events)

        print("Loaded data for", len(eval_active_X), "subjects.")

        if(n_classes==2):
            eval_active_X = [x[np.isin(y, [769, 770])] for x, y in zip(eval_active_X, eval_active_y)]
            eval_active_y = [y[np.isin(y, [769, 770])] for y in eval_active_y]
        
        train_active_y = encode_labels(train_active_y)
        eval_active_y = encode_labels(eval_active_y)

        align_per_class = 14
        n_subjects = 9
        if(n_classes==2):
            classes = [0, 1]
        else:
            classes = [0, 1, 2, 3]  # Two classes as per your setup
        epsilon = 1e-6
        n_channels = 22
        accuracies = []

        for subj_idx in range(len(train_active_X)):
            # print(subj_idx)
            # Split data into train/test using leave-one-subject-out
            X_target = eval_active_X[subj_idx]
            y_target = eval_active_y[subj_idx]
            
            # Concatenate data from other subjects
            X_source = train_active_X[subj_idx]
            y_source = train_active_y[subj_idx]

            cov_estimator = Covariances(estimator='scm') 
            X_source = cov_estimator.fit_transform(X_source) 
            X_target = cov_estimator.fit_transform(X_target) 

            # Split target data into alignment and test sets
            align_indices = []
            test_indices = []
            for cls in classes:
                cls_indices = np.where(y_target == cls)[0]
                np.random.shuffle(cls_indices)
                align_indices.extend(cls_indices[:align_per_class])
                test_indices.extend(cls_indices[align_per_class:])

            M_source = mean_riemann(X_source)
            M_target_align = mean_riemann(X_target[align_indices])

            M_source_inv_half = np.linalg.inv(scipy.linalg.sqrtm(M_source))
            source_rct = [M_source_inv_half @ C @ M_source_inv_half for C in X_source]
            
            M_target_inv_half = np.linalg.inv(scipy.linalg.sqrtm(M_target_align))
            target_rct = [M_target_inv_half @ C @ M_target_inv_half for C in X_target]
            

            # Compute dispersion d for source
            print("Calculating source dispersions")
            d = 0
            for C_ret in source_rct:
                A_inv_half = np.linalg.inv(scipy.linalg.sqrtm(np.eye(n_channels)))
                C = np.dot(np.dot(A_inv_half, C_ret), A_inv_half)
                log_C = scipy.linalg.logm(C)
                d += np.linalg.norm(log_C, 'fro')**2

            # Compute dispersion tilde_d for T_l
            print("Calculating target dispersions")
            target_align_rct = [target_rct[i] for i in align_indices]
            tilde_d = 0
            for C_ret in target_align_rct:
                A_inv_half = np.linalg.inv(scipy.linalg.sqrtm(np.eye(n_channels)))
                C = np.dot(np.dot(A_inv_half, C_ret), A_inv_half)
                log_C = scipy.linalg.logm(C)
                tilde_d += np.linalg.norm(log_C, 'fro')**2

            # Compute scaling factor s
            s = np.sqrt(d / tilde_d)

            # Stretch target matrices
            target_str = [scipy.linalg.fractional_matrix_power(C_ret, s) for C_ret in target_rct]

            # Compute class means for source
            print("Calculating source class means")
            M_k = []
            for cls in classes:
                source_cls_rct = np.stack([source_rct[i] for i in range(len(y_source)) if y_source[i] == cls])
                source_mean_cls = mean_riemann(source_cls_rct, tol=1e-6, maxiter=100)
                M_k.append(source_mean_cls)

            # Compute class means for labeled target
            print("Calculating labelled target class means")
            tilde_M_k = []
            for cls in classes:
                align_cls_str = np.stack([target_str[i] for i in range(len(y_target[align_indices])) if y_target[align_indices][i] == cls])
                align_mean_cls = mean_riemann(align_cls_str, tol=1e-6, maxiter=100)
                tilde_M_k.append(align_mean_cls)

            # Optimize for U (rotation matrix)
            # Parameterize U = exp(A) where A is skew-symmetric
            manifold = SpecialOrthogonalGroup(n_channels)

            # Define the cost function
            @pymanopt.function.autograd(manifold)
            def cost(U):
                total = 0.0
                for k in range(len(classes)):
                    tilde_M_k_inv_sqrt = linalg.inv(linalg.sqrtm(tilde_M_k[k]))
                    transformed = anp.dot(anp.dot(U, M_k[k]), U.T)
                    arg_logm = anp.dot(anp.dot(tilde_M_k_inv_sqrt, transformed), tilde_M_k_inv_sqrt)
                    log_term = logm_approx(arg_logm)
                    total += frobenius_norm(log_term) ** 2
                return total

            # Create the optimization problem
            problem = Problem(manifold=manifold, cost=cost)

            # Choose a solver and run it
            solver = SteepestDescent()
            U_opt = solver.run(problem)

            # Rotate target matrices
            U_opt = np.array(U_opt.point)
            target_rot = [np.dot(np.dot(U_opt.T, C_str), U_opt) for C_str in target_str]


            X_train = np.concatenate((np.array(source_rct), np.array([target_rot[i] for i in align_indices])))
            y_train = np.concatenate((y_source, y_target[align_indices]))

            mdm = MDM(metric='riemann')
            mdm.fit(X_train, y_train)

            X_test = np.array([target_rot[i] for i in test_indices])
            y_pred = mdm.predict(X_test)

            # Compute and print accuracy
            accuracy = accuracy_score(y_target[test_indices], y_pred)
            accuracies.append(accuracy)
        print('tmin: ', tmin, ' tmax: ',tmax)
        for subj_idx in range(len(train_active_X)):
            print(f"Subject {subj_idx+1} Test Accuracy: {accuracies[subj_idx]:.2f}")

        print(f"\nMean Cross-Validation Accuracy: {np.mean(accuracies):.2f} ± {np.std(accuracies):.2f}")
    

        
    

In [11]:
twofour_crosssession(4)

/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770', '771', '772']
Subject 1: Epoch data shape (288, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770', '771', '772']
Subject 2: Epoch data shape (288, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


...

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 8: Epoch data shape (288, 22, 251)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 9: Epoch data shape (288, 22, 251)
Loaded data for 9 subjects.
Calculating source dispersions
Calculating target dispersions
Calculating source class means
Calculating labelled target class means
Optimizing...
Iteration    Cost                       Gradient norm     
---------    -----------------------    --------------    
   1         +1.5408486276005148e+01    1.18827145e+00    
   2         +1.4373357359609393e+01    1.12771995e+00    
   3         +1.3489903737708833e+01    2.51933045e+00    
   4         +1.2941827814585471e+01    1.31235115e+00    
   5         +1.2118483543311505e+01    1.33243474e+00    
   6         +1.2034883108489250e+01    1.13081667e+00    
   7         +1.1745962144372275e+01    8.93548114e-01    
   8         +1.1493249662507655e+01    6.80138804e-01    
   9         +1.1404937561808952e+01    5.47081983e-01    
  10         +1.1

/tmp/ipykernel_17143/633717832.py:185: RuntimeWarning: logm result may be inaccurate, approximate err = 2.539456510449916e-13
  log_C = scipy.linalg.logm(C)


Calculating source class means
Calculating labelled target class means
Optimizing...
Iteration    Cost                       Gradient norm     
---------    -----------------------    --------------    
   1         +4.7532617833548706e+01    3.01976718e+01    
   2         +3.2099004138161249e+01    1.32009412e+01    
   3         +2.9910618513336040e+01    1.61832092e+01    
   4         +2.4657614814650941e+01    7.08604217e+00    
   5         +2.3322323620130433e+01    7.24616077e+00    
   6         +2.2620692728625833e+01    5.62165753e+00    
   7         +2.2157194208885553e+01    3.76862018e+00    
   8         +2.1940090400838038e+01    4.49181500e+00    
   9         +2.1599964409467177e+01    2.86826560e+00    
  10         +2.1438463177703088e+01    4.27909131e+00    
  11         +2.1122485726805053e+01    1.96996980e+00    
  12         +2.0879550894498944e+01    4.18205650e+00    
  13         +2.0732288024257716e+01    3.75923482e+00    
  14         +2.05134850447769